<a href="https://colab.research.google.com/github/findsamirks-commits/retail-demand-forecasting-api/blob/main/retail_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 Retail Sales Forecasting Lab
Welcome to our forecasting lab! In this notebook, we are going to act like inventory and demand planners. We will teach a machine learning model to look at past sales trends (history) and forecast future demand so stores don't run out of stock or over-order!

## Step 1: Data Loading & Feature Engineering
Before an AI can predict the future, it needs to understand the past. In retail, raw numbers alone aren't enough—we need to engineer features like **lagged sales** (what sold last month) and **rolling averages** (short-term trends) so the algorithm can recognize patterns.

In [1]:
import pandas as pd
import numpy as np

# MAGIC DATA LOADING: Simulating historical retail sales transactions
# In a real retail setup, this would load your daily store or SKU sales logs.
url = "https://raw.githubusercontent.com/selva86/datasets/master/a10.csv"
df = pd.read_csv(url)

# Rename columns for clarity
df.columns = ['date', 'sales_value']
df['date'] = pd.to_datetime(df['date'])

# Feature Engineering: Creating lag features (sales from previous months) to help the AI learn patterns
df['lag_1_month'] = df['sales_value'].shift(1)
df['rolling_mean_3'] = df['sales_value'].rolling(window=3).mean()

# Drop missing values created by shifting/rolling
df = df.dropna()

print(f"Dataset successfully loaded and engineered! Shape: {df.shape}")
display(df.head())

Dataset successfully loaded and engineered! Shape: (202, 4)


,date,sales_value,lag_1_month,rolling_mean_3
2,1991-09-01,3.252221,3.180891,3.319901
3,1991-10-01,3.611003,3.252221,3.348038
4,1991-11-01,3.565869,3.611003,3.476364
5,1991-12-01,4.306371,3.565869,3.827748
6,1992-01-01,5.088335,4.306371,4.320192


## Step 2: Training the Machine Learning Model (Random Forest)
Now that our data has history and trends built into it, we will split our data into training and testing sets. We use a **Random Forest Regressor** to learn the mathematical relationships between past sales patterns and future outcomes, then grade its performance using Root Mean Squared Error (RMSE).

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Define Features (X) and Target (y)
# We want the model to learn using the date parts, lag, and rolling mean to predict 'sales_value'
X = df[['lag_1_month', 'rolling_mean_3']]
y = df['sales_value']

# 2. Split into Training (80%) and Testing (20%) sets
# Note: For strict time series, sequential splitting is common, but train_test_split works great for learning the basics!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and train the Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Evaluate the model
predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Model successfully trained!")
print(f"Validation Root Mean Squared Error (RMSE): {rmse:.4f}")

Model successfully trained!
Validation Root Mean Squared Error (RMSE): 1.4629


## Step 3: Serializing (Saving) the AI Brain
To use our trained model outside of this notebook (like in a web app or backend server), we must save it to disk as a `.joblib` file. This preserves all the patterns the Random Forest learned.

In [3]:
import joblib
import os

# 1. Create a directory for our models
os.makedirs('models', exist_ok=True)
model_path = 'models/retail_demand_model.joblib'

# 2. Save the trained model
joblib.dump(model, model_path)
print(f"Success! Retail demand model saved to: {model_path}")

Success! Retail demand model saved to: models/retail_demand_model.joblib


## Step 4: Creating the FastAPI Backend (`app.py`)
Instead of running a manual terminal command, we can use Colab's `%%writefile` magic command to automatically generate our `app.py` script right here in our workspace.

In [4]:
%%writefile app.py
from fastapi import FastAPI
import joblib
import pandas as pd
from pydantic import BaseModel

# Initialize FastAPI app
app = FastAPI(title="Retail Demand Forecasting API", version="1.0")

# Load the trained Random Forest model
model = joblib.load('models/retail_demand_model.joblib')

# Define the expected input data structure using Pydantic
class RetailInput(BaseModel):
    lag_1_month: float
    rolling_mean_3: float

@app.get("/")
def home():
    return {"message": "Welcome to the Retail Demand Forecasting API! Use the /predict endpoint to get sales forecasts."}

@app.post("/predict")
def predict_demand(data: RetailInput):
    # Convert incoming JSON data into a Pandas DataFrame
    input_df = pd.DataFrame([{
        'lag_1_month': data.lag_1_month,
        'rolling_mean_3': data.rolling_mean_3
    }])

    # Generate prediction
    prediction = model.predict(input_df)[0]

    return {
        "input_features": data.dict(),
        "predicted_sales_demand": round(float(prediction), 4)
    }

Writing app.py


## Step 5: Launching the API
We will now install the required server packages and launch our FastAPI backend using Uvicorn, creating a live bridge to our trained retail model.

In [5]:
# Install required backend servers and tunneling tools
!pip install -q fastapi uvicorn pyngrok

# Run FastAPI in the background on port 8000
get_ipython().system_raw('uvicorn app:app --host 0.0.0.0 --port 8000 &')

print("FastAPI backend is running in the background!")
print("You can now build a frontend or test the endpoints locally or via documentation.")

FastAPI backend is running in the background!
You can now build a frontend or test the endpoints locally or via documentation.
